# Hermes Agent - Jupyter Notebook

This notebook allows you to interact with Hermes, the NousResearch AI agent, directly from Jupyter.

## ⚠️ Supabase RLS Setup Required!

Before uploading, run this SQL in Supabase SQL Editor:
```sql
-- SELECT: allow reading
CREATE POLICY "public_select" ON storage.objects FOR SELECT TO PUBLIC USING (bucket_id = 'manini');

-- INSERT: allow uploads
CREATE POLICY "public_insert" ON storage.objects FOR INSERT TO PUBLIC WITH CHECK (bucket_id = 'manini');

-- UPDATE: allow updates
CREATE POLICY "public_update" ON storage.objects FOR UPDATE TO PUBLIC USING (bucket_id = 'manini') WITH CHECK (bucket_id = 'manini');
```

In [ ]:
# Install Supabase client
!pip install --break-system-packages supabase

In [ ]:
# Setup Supabase sync
import os
import json
from supabase import create_client, Client

# Credentials
SUPABASE_URL = os.environ.get('SUPABASE_URL', 'https://opdpexsytsaldlworztz.supabase.co')
SUPABASE_KEY = os.environ.get('SUPABASE_KEY', '')  # anon key
BUCKET_NAME = os.environ.get('SUPABASE_BUCKET', 'manini')
HERMES_REMOTE = os.environ.get('HERMES_REMOTE', '')  # URL to backup hermes config

NOTEBOOK_DIR = '/data/notebooks'
os.makedirs(NOTEBOOK_DIR, exist_ok=True)
os.chdir(NOTEBOOK_DIR)

# Hermes paths
HERMES_HOME = os.path.expanduser('~/.hermes')
CONFIG_PATH = os.path.join(HERMES_HOME, 'config.yaml')
MEMORY_PATH = os.path.join(HERMES_HOME, 'memory')

print(f"Hermes Home: {HERMES_HOME}")
print(f"Config: {CONFIG_PATH}")
print(f"Memory: {MEMORY_PATH}")

# Initialize Supabase
if SUPABASE_URL and SUPABASE_KEY:
    supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
    print("\n✓ Connected to Supabase!")
else:
    supabase = None
    print("\n⚠️  Set SUPABASE_KEY for notebook sync")

## 📥 Download Hermes Config & Memory from Supabase

In [ ]:
def download_hermes_config():
    """Download hermes config and memory from Supabase."""
    if not supabase:
        print("Supabase not configured")
        return
    
    os.makedirs(HERMES_HOME, exist_ok=True)
    files_downloaded = []
    
    try:
        files = supabase.storage.from_(BUCKET_NAME).list()
        
        for file in files:
            name = file.get('name', '')
            if name.startswith('hermes/') and name.endswith(('.yaml', '.json', '.md')):
                print(f"Downloading {name}...")
                data = supabase.storage.from_(BUCKET_NAME).download(name)
                # Extract relative path
                rel_path = name.replace('hermes/', '', 1)
                save_path = os.path.join(HERMES_HOME, rel_path)
                os.makedirs(os.path.dirname(save_path), exist_ok=True)
                with open(save_path, 'wb') as f:
                    f.write(data)
                files_downloaded.append(rel_path)
        
        if files_downloaded:
            print(f"\n✓ Downloaded {len(files_downloaded)} files to {HERMES_HOME}")
        else:
            print("\nNo hermes config found in storage")
            
    except Exception as e:
        print(f"Error: {e}")

download_hermes_config()

## 📤 Upload Hermes Config & Memory to Supabase

In [ ]:
def upload_hermes_config():
    """Upload hermes config and memory to Supabase."""
    if not supabase:
        print("Supabase not configured")
        return
    
    if not os.path.exists(HERMES_HOME):
        print(f"No Hermes config at {HERMES_HOME}")
        return
    
    uploaded = 0
    
    for root, dirs, files in os.walk(HERMES_HOME):
        for filename in files:
            if filename.endswith(('.yaml', '.json', '.md', '.txt')):
                file_path = os.path.join(root, filename)
                remote_name = f"hermes/{os.path.relpath(file_path, HERMES_HOME)}"
                print(f"Uploading {remote_name}...")
                try:
                    with open(file_path, 'rb') as f:
                        supabase.storage.from_(BUCKET_NAME).upload(
                            remote_name,
                            f.read(),
                            {"contentType": "text/plain"}
                        )
                    uploaded += 1
                except Exception as e:
                    try:
                        with open(file_path, 'rb') as f:
                            supabase.storage.from_(BUCKET_NAME).update(
                                remote_name,
                                f.read(),
                                {"contentType": "text/plain"}
                            )
                        uploaded += 1
                    except Exception as e2:
                        print(f"  Error: {e2}")
    
    print(f"\n✓ Uploaded {uploaded} files to Supabase")

upload_hermes_config()

## 📤 Download Notebooks from Supabase

In [ ]:
def pull_notebooks():
    """Download notebooks from Supabase."""
    if not supabase:
        print("Supabase not configured")
        return
    
    try:
        files = supabase.storage.from_(BUCKET_NAME).list()
        
        for file in files:
            name = file.get('name', '')
            if name.endswith('.ipynb'):
                print(f"Downloading {name}...")
                data = supabase.storage.from_(BUCKET_NAME).download(name)
                with open(name, 'wb') as f:
                    f.write(data)
        
        print(f"\n✓ Done! Files: {os.listdir('.')}")
        
    except Exception as e:
        print(f"Error: {e}")

pull_notebooks()

## ⬆️ Upload Notebooks to Supabase

In [ ]:
def upload_notebooks():
    """Upload notebooks to Supabase."""
    if not supabase:
        print("Supabase not configured")
        return
    
    uploaded = 0
    
    for filename in os.listdir('.'):
        if filename.endswith('.ipynb'):
            print(f"Uploading {filename}...")
            try:
                with open(filename, 'rb') as f:
                    supabase.storage.from_(BUCKET_NAME).upload(
                        filename, f.read(), {"contentType": "application/json"})
                uploaded += 1
            except Exception as e:
                try:
                    with open(filename, 'rb') as f:
                        supabase.storage.from_(BUCKET_NAME).update(
                            filename, f.read(), {"contentType": "application/json"})
                    uploaded += 1
                except Exception as e2:
                    print(f"  Error: {e2}")
    
    print(f"\n✓ {uploaded} notebooks synced")

upload_notebooks()

---

## 🚀 Install Hermes Agent

In [ ]:
!pip install --break-system-packages git+https://github.com/NousResearch/hermes-agent.git

## Initialize Hermes Agent

In [ ]:
from run_agent import AIAgent

# Set your API key:
# os.environ['OPENAI_API_KEY'] = 'your-key'

agent = AIAgent(
    model="openai/gpt-4o",
    quiet_mode=True,
)

print("Hermes Agent initialized!")

## Chat with Hermes

In [ ]:
response = agent.chat("Hello!")
print(response)